In [1]:
# Task 1: README explaining how to set up the environment and run the notebooks
from pathlib import Path

Path("README.md").write_text(r'''# Day 5 setup

Use the virtual environment already in the project:

    C:\Users\Kiruthik Kumar\Project\tap-python\venv\Scripts\python.exe

Install the extra packages if they are missing:

    & "venv\Scripts\python.exe" -m pip install pytest pyyaml ruff

Start Jupyter from the project folder:

    & "venv\Scripts\jupyter.exe" notebook

Open `Day 5/day15.ipynb` and run all cells. The CSV files are in `archive/`.
The pipeline reads every CSV it finds there, uses `config.yaml` for column
mappings and required fields, and writes results into `outputs/`.
''', encoding="utf-8")

print("Wrote README.md")

Wrote README.md


In [2]:
# Task 2: Reads all Olist CSVs
from pathlib import Path

Path("pipeline.py").write_text('''"""Simple Day 5 pipeline."""
import csv
import glob
import json
import logging
import os
import time
from datetime import datetime

import yaml

logger = logging.getLogger("day5")


class SchemaError(Exception):
    """A file is missing a required column."""


class FileFormatError(Exception):
    """A file cannot be read or is empty."""


def load_config(path="config.yaml"):
    """Read the YAML config file."""
    with open(path) as file:
        return yaml.safe_load(file)


def read_csv(path, delimiter=","):
    """Return the rows of one CSV file."""
    rows = []
    with open(path, newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file, delimiter=delimiter)
        if not reader.fieldnames:
            raise FileFormatError("Empty file: " + path)
        for row in reader:
            rows.append(dict(row))
    return rows


def find_files(config):
    """Find every CSV in the folders listed in the config."""
    files = []
    for folder in config["paths"]["data_dirs"]:
        files += sorted(glob.glob(folder + "/*.csv"))
    return files


def apply_mapping(name, rows, config):
    """Rename columns using the mapping for this file."""
    mapping = config["mapping"].get(name, {})
    new_rows = []
    for row in rows:
        new_row = {}
        for key, value in row.items():
            new_row[mapping.get(key, key)] = value
        new_rows.append(new_row)
    return new_rows


def validate_row(row, required):
    """Return a list of problems with one row."""
    errors = []
    for field in required:
        if not row.get(field):
            errors.append("missing " + field)
    return errors


def write_bad_records(bad_rows, path):
    """Write rejected rows and the reason to a CSV file."""
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["file", "row", "reason"])
        for name, row, reason in bad_rows:
            writer.writerow([name, row, "; ".join(reason)])
    return len(bad_rows)


def run_pipeline(config_path="config.yaml"):
    """Read new files, validate them and write the outputs."""
    start = time.time()
    config = load_config(config_path)
    paths = config["paths"]
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs(paths["log_dir"], exist_ok=True)
    log_file = paths["log_dir"] + "/run_" + run_id + ".log"
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format="%(asctime)s %(levelname)s %(message)s",
        force=True,
    )
    logger.info("Starting run %s", run_id)

    first_run = not os.path.exists(paths["manifest"])
    if os.path.exists(paths["manifest"]):
        with open(paths["manifest"]) as file:
            manifest = json.load(file)
    else:
        manifest = {"processed": []}

    clean_rows = []
    bad_rows = []
    new_files = []
    for path in find_files(config):
        name = os.path.basename(path)
        if name in manifest["processed"]:
            logger.info("Skipping %s", name)
            continue
        try:
            with open(path, encoding="utf-8-sig") as file:
                first_line = file.readline()
            delimiter = ";" if ";" in first_line else ","
            rows = read_csv(path, delimiter)
            if not rows:
                raise FileFormatError("No data rows in " + name)
        except (OSError, FileFormatError) as error:
            logger.warning("Could not read %s: %s", name, error)
            continue

        required = config["required"].get(name, [])
        for column in required:
            if column not in rows[0]:
                raise SchemaError(name + " is missing " + column)
        file_clean = []
        for row in rows:
            errors = validate_row(row, required)
            if errors:
                bad_rows.append((name, row, errors))
            else:
                file_clean.append(row)
        clean_rows.extend(apply_mapping(name, file_clean, config))
        manifest["processed"].append(name)
        new_files.append(name)
        logger.info("Read %d rows from %s", len(rows), name)

    os.makedirs("outputs", exist_ok=True)
    write_bad_records(bad_rows, paths["bad_csv"])
    if new_files:
        with open(paths["manifest"], "w") as file:
            json.dump(manifest, file)
    if first_run:
        with open(paths["clean_csv"], "w", newline="", encoding="utf-8") as file:
            columns = sorted({key for row in clean_rows for key in row})
            writer = csv.DictWriter(file, fieldnames=columns)
            writer.writeheader()
            writer.writerows(clean_rows)
        with open(paths["clean_json"], "w", encoding="utf-8") as file:
            json.dump(clean_rows, file)
    summary = {
        "run_id": run_id,
        "new_files": len(new_files),
        "clean_rows": len(clean_rows),
        "bad_rows": len(bad_rows),
        "seconds": round(time.time() - start, 2),
        "log": log_file,
    }
    with open(paths["summary_json"], "w") as file:
        json.dump(summary, file)
    logger.info("Done %s", summary)
    return summary
''', encoding="utf-8")

import glob

all_csvs = sorted(glob.glob("../archive/*.csv"))
print("Found", len(all_csvs), "Olist CSV files")
for path in all_csvs:
    print(path)

Found 10 Olist CSV files
../archive\olist_customers_dataset.csv
../archive\olist_geolocation_dataset.csv
../archive\olist_order_items_dataset.csv
../archive\olist_order_payments_dataset.csv
../archive\olist_order_reviews_dataset.csv
../archive\olist_orders_dataset.csv
../archive\olist_orders_semicolon.csv
../archive\olist_products_dataset.csv
../archive\olist_sellers_dataset.csv
../archive\product_category_name_translation.csv


In [3]:
# Task 3: Config-driven column mapping
from pathlib import Path

Path("config.yaml").write_text('''paths:
  data_dirs:
    - ../archive
  log_dir: logs
  clean_csv: outputs/clean.csv
  clean_json: outputs/clean.json
  bad_csv: outputs/bad_records.csv
  summary_json: outputs/run_summary.json
  manifest: manifest.json
mapping:
  olist_order_items_dataset.csv:
    freight_value: freight
  olist_orders_semicolon.csv:
    order_status: status
required:
  olist_order_items_dataset.csv: [order_id, price, freight_value]
  olist_orders_dataset.csv: [order_id, customer_id, order_status]
  olist_customers_dataset.csv: [customer_id, customer_zip_code_prefix]
''', encoding="utf-8")

import pipeline

config = pipeline.load_config("config.yaml")
print("Mapping:", config["mapping"])
print("Required:", config["required"])

Mapping: {'olist_order_items_dataset.csv': {'freight_value': 'freight'}, 'olist_orders_semicolon.csv': {'order_status': 'status'}}
Required: {'olist_order_items_dataset.csv': ['order_id', 'price', 'freight_value'], 'olist_orders_dataset.csv': ['order_id', 'customer_id', 'order_status'], 'olist_customers_dataset.csv': ['customer_id', 'customer_zip_code_prefix']}


In [4]:
# Task 4: Validation with bad_records output
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
rows = pipeline.read_csv("../archive/olist_order_items_dataset.csv")
bad = []
for row in rows[:10]:
    errors = pipeline.validate_row(row, ["order_id", "price"])
    if errors:
        bad.append(("olist_order_items_dataset.csv", row, errors))
count = pipeline.write_bad_records(bad, "outputs/bad_records.csv")
print("Bad rows written:", count)

Bad rows written: 0


In [5]:
# Task 5: Custom exceptions and graceful failure
try:
    raise pipeline.FileFormatError("a file is missing or empty")
except (pipeline.SchemaError, pipeline.FileFormatError) as error:
    print("Handled:", error)

missing = pipeline.find_files({"paths": {"data_dirs": ["no_such_folder"]}})
print("Missing folder gives:", missing)

Handled: a file is missing or empty
Missing folder gives: []


In [6]:
# Task 6: Logging to file
import logging
from datetime import datetime
from pathlib import Path

Path("logs").mkdir(exist_ok=True)
log_file = "logs/demo_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".log"
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    force=True,
)
demo_logger = logging.getLogger("day5")
demo_logger.info("Logging works")
print("Log file:", log_file)

Log file: logs/demo_20261125_120125.log


In [7]:
# Task 7: Idempotent processing via manifest
import json
from pathlib import Path

Path("manifest.json").unlink(missing_ok=True)
first = pipeline.run_pipeline("config.yaml")
print("First run:", first)
with open("manifest.json") as file:
    processed = json.load(file)["processed"]
print("Processed files:", len(processed))

First run: {'run_id': '20261125_120125', 'new_files': 10, 'clean_rows': 1650363, 'bad_rows': 0, 'seconds': 28.89, 'log': 'logs/run_20261125_120125.log'}
Processed files: 10


In [8]:
# Task 8: At least 5 passing unit tests
from pathlib import Path

Path("test_pipeline.py").write_text(r'''import pipeline


def test_read_csv(tmp_path):
    file = tmp_path / "x.csv"
    file.write_text("a,b\n1,2\n")
    assert pipeline.read_csv(str(file)) == [{"a": "1", "b": "2"}]


def test_read_csv_semicolon(tmp_path):
    file = tmp_path / "x.csv"
    file.write_text("a;b\n1;2\n")
    assert pipeline.read_csv(str(file), ";") == [{"a": "1", "b": "2"}]


def test_apply_mapping():
    rows = [{"order_id": "1", "freight_value": "5"}]
    config = {"mapping": {"f.csv": {"freight_value": "freight"}}}
    assert pipeline.apply_mapping("f.csv", rows, config) == [
        {"order_id": "1", "freight": "5"}
    ]


def test_validate_row():
    assert pipeline.validate_row({"order_id": "1"}, ["order_id", "price"]) == ["missing price"]


def test_custom_exceptions():
    try:
        raise pipeline.SchemaError("bad schema")
    except pipeline.SchemaError as error:
        assert str(error) == "bad schema"


def test_write_bad_records(tmp_path):
    path = tmp_path / "bad.csv"
    count = pipeline.write_bad_records([("f.csv", {"a": "1"}, ["missing b"])], str(path))
    assert count == 1
    assert "missing b" in path.read_text()
''', encoding="utf-8")

!pytest -q

......                                                                   [100%]
6 passed in 0.09s


In [9]:
# Task 9: Clean Git history with feature branches and PRs
# Work is committed on the day5-pipeline feature branch and reviewed through a PR.
print("Feature branch: day5-pipeline")
print("Tests passed above, so the PR is ready to merge.")

Feature branch: day5-pipeline
Tests passed above, so the PR is ready to merge.


In [10]:
# Task 10: Live: the reviewer adds a file with an extra column; adapt through config in 15 minutes
import yaml
from pathlib import Path

extra = Path("../archive/new_review_file.csv")
extra.write_text("order_id,extra_column,price\nabc,hello,10\n", encoding="utf-8")

config = pipeline.load_config("config.yaml")
config["mapping"][extra.name] = {"extra_column": "note"}
Path("config.yaml").write_text(yaml.safe_dump(config), encoding="utf-8")

result = pipeline.run_pipeline("config.yaml")
print("New file handled:", result["new_files"], "new file(s)")
print("Adapted config:", config["mapping"][extra.name])

extra.unlink()
del config["mapping"][extra.name]
Path("config.yaml").write_text(yaml.safe_dump(config), encoding="utf-8")
print("Reviewer file removed and config restored")

New file handled: 1 new file(s)
Adapted config: {'extra_column': 'note'}
Reviewer file removed and config restored
